# Implementación de Modelos No Supervisados (CRISP-DM)

Este notebook corresponde a la **Evaluación Parcial 3**, donde se implementan modelos de Machine Learning no supervisados siguiendo la metodología **CRISP-DM**. Se utilizan los datos del set *Video Game Sales* para identificar grupos naturales de videojuegos basados en variables como ventas, críticas y puntuaciones de usuarios.

Las etapas cubiertas en este informe incluyen:

- Comprensión del negocio  
- Comprensión de los datos  
- Preparación de datos  
- Modelado no supervisado  
- Evaluación de modelos  
- Interpretación y segmentación final

El objetivo es **determinar el número óptimo de clusters**, evaluar su coherencia y presentar insights que puedan usarse para decisiones estratégicas en la industria de videojuegos.



In [ ]:
# ==============================
# 1 Imports y carga de datos
# ==============================
import warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

# Visualización solo con Plotly
import plotly.express as px

# ML / Clustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)

# UMAP opcional (no es obligatorio para la rúbrica)
try:
    import umap
except Exception:
    umap = None

# Ruta fija del dataset (la que tú dijiste)
DATA_PATH = Path(r"C:\Users\lttlk\Documents\Nueo\machinegame\data\01_raw\Video_Games_Sales_as_at_22_Dec_2016.csv")

assert DATA_PATH.exists(), f"No existe el CSV en {DATA_PATH}"

df_raw = pd.read_csv(DATA_PATH)
print("Shape raw:", df_raw.shape)
df_raw.head()


Shape raw: (16719, 16)


,Name,Platform,Year_of_Release,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,Developer,Rating
0,Wii Sports,Wii,2006.0,Sports,Nintendo,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8.0,322.0,Nintendo,E
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24,NaN,NaN,NaN,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,Nintendo,E
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8.0,192.0,Nintendo,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37,NaN,NaN,NaN,NaN,NaN,NaN


## Comprensión de los Datos

El dataset utilizado proviene del archivo:  
**Video_Games_Sales_as_at_22_Dec_2016.csv**

Incluye información de miles de videojuegos publicados en distintos años, con variables numéricas relacionadas a:

- Ventas por región (NA, EU, JP, Other, Global)
- Puntajes de críticos (Critic_Score, Critic_Count)
- Puntajes de usuarios (User_Score, User_Count)
- Año de lanzamiento

Se eliminan columnas no numéricas y filas con valores faltantes, con el propósito de construir matrices limpias y consistentes para aplicar algoritmos no supervisados.


In [ ]:
# ==============================
# 2 EDA rápido del dataset
# ==============================

print("Columnas del dataset:")
print(df_raw.columns.tolist())

print("\nInfo:")
print(df_raw.info())

print("\nDescripción numérica:")
df_raw.describe(include="number").T.head(15)


Columnas del dataset:
['Name', 'Platform', 'Year_of_Release', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales', 'Critic_Score', 'Critic_Count', 'User_Score', 'User_Count', 'Developer', 'Rating']

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16719 entries, 0 to 16718
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16717 non-null  object 
 1   Platform         16719 non-null  object 
 2   Year_of_Release  16450 non-null  float64
 3   Genre            16717 non-null  object 
 4   Publisher        16665 non-null  object 
 5   NA_Sales         16719 non-null  float64
 6   EU_Sales         16719 non-null  float64
 7   JP_Sales         16719 non-null  float64
 8   Other_Sales      16719 non-null  float64
 9   Global_Sales     16719 non-null  float64
 10  Critic_Score     8137 non-null   float64
 11  Critic_Count     8137 non-null   float64
 12  U

,count,mean,std,min,25%,50%,75%,max
Year_of_Release,16450.0,2006.487356,5.878995,1980.00,2003.00,2007.00,2010.00,2020.00
NA_Sales,16719.0,0.263330,0.813514,0.00,0.00,0.08,0.24,41.36
EU_Sales,16719.0,0.145025,0.503283,0.00,0.00,0.02,0.11,28.96
JP_Sales,16719.0,0.077602,0.308818,0.00,0.00,0.00,0.04,10.22
Other_Sales,16719.0,0.047332,0.186710,0.00,0.00,0.01,0.03,10.57
Global_Sales,16719.0,0.533543,1.547935,0.01,0.06,0.17,0.47,82.53
Critic_Score,8137.0,68.967679,13.938165,13.00,60.00,71.00,79.00,98.00
Critic_Count,8137.0,26.360821,18.980495,3.00,12.00,21.00,36.00,113.00
User_Score,7590.0,7.125046,1.500006,0.00,6.40,7.50,8.20,9.70
User_Count,7590.0,162.229908,561.282326,4.00,10.00,24.00,81.00,10665.00


In [ ]:
# ==============================
# 3 selección de numéricas y limpieza
# ==============================

# Copia de trabajo
df = df_raw.copy()

# Nos quedamos solo con columnas numéricas
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
print("Columnas numéricas detectadas:", numeric_cols)

# Opcional: intentar descartar columnas claramente de identificador si existen
drop_id_like = []
for c in numeric_cols:
    cl = c.lower()
    if "rank" in cl or "id" in cl:
        drop_id_like.append(c)

if drop_id_like:
    print("Columnas descartadas por parecer IDs:", drop_id_like)
    numeric_cols = [c for c in numeric_cols if c not in drop_id_like]

# DataFrame solo con numéricas seleccionadas
df_num = df[numeric_cols].copy()

# Eliminamos filas con NA en esas columnas
before = df_num.shape[0]
df_num = df_num.dropna()
after = df_num.shape[0]
print(f"Filas antes: {before}, después de dropna: {after}")

df_num.head()


Columnas numéricas detectadas: ['Year_of_Release', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales', 'Critic_Score', 'Critic_Count', 'User_Score', 'User_Count']
Filas antes: 16719, después de dropna: 6894


,Year_of_Release,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count
0,2006.0,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8.0,322.0
2,2008.0,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0
3,2009.0,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8.0,192.0
6,2006.0,11.28,9.14,6.50,2.88,29.80,89.0,65.0,8.5,431.0
7,2006.0,13.96,9.18,2.93,2.84,28.92,58.0,41.0,6.6,129.0


In [ ]:
# ==============================
# 4 escalado + PCA
# ==============================

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_num)

print("Shape de X_scaled:", X_scaled.shape)

# PCA a 2 componentes para visualización
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Varianza explicada por componente:", pca.explained_variance_ratio_)

# Creamos un DataFrame con las componentes
df_pca = pd.DataFrame(X_pca, columns=["PC1", "PC2"], index=df_num.index)
df_pca.head()


Shape de X_scaled: (6894, 10)
Varianza explicada por componente: [0.42075619 0.16529468]


,PC1,PC2
0,74.004319,-23.702399
2,33.186567,-8.812657
3,30.044032,-8.213225
6,29.627469,-7.300756
7,25.882295,-9.198088


## Reducción de Dimensionalidad con PCA

Para visualizar correctamente los clusters, se aplicó **PCA** sobre los datos estandarizados.  
Los dos primeros componentes principales explican:

- **PC1:** 42.07% de la varianza  
- **PC2:** 16.52% de la varianza

Esto indica que la estructura del dataset puede representarse adecuadamente en un plano 2D manteniendo más del **58%** de la información original. Esta proyección se utilizará en los gráficos de clustering para facilitar la interpretación visual.


In [ ]:
# ==============================
# 5 búsqueda de k óptimo (KMeans)
# ==============================

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

results = []

K_RANGE = range(2, 11)  # k = 2 a 10

for k in K_RANGE:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(X_scaled)

    sil = silhouette_score(X_scaled, labels)
    ch = calinski_harabasz_score(X_scaled, labels)
    db = davies_bouldin_score(X_scaled, labels)

    results.append({
        "k": k,
        "silhouette": sil,
        "calinski_harabasz": ch,
        "davies_bouldin": db
    })

df_kmeans_metrics = pd.DataFrame(results)
df_kmeans_metrics


,k,silhouette,calinski_harabasz,davies_bouldin
0,2,0.717705,1746.742218,1.131995
1,3,0.519580,1596.542975,1.204379
2,4,0.220820,1674.645568,1.305086
3,5,0.216470,1613.243633,1.334667
4,6,0.207476,1498.022979,1.308411
5,7,0.212412,1473.855205,1.274452
6,8,0.185347,1376.105474,1.317187
7,9,0.185677,1454.978809,1.191772
8,10,0.171723,1355.163366,1.264239


In [ ]:
# ==============================
# 6 Elegir k óptimo y ajustar KMeans final
# ==============================

# Elegimos k por mejor silhouette
best_row = df_kmeans_metrics.sort_values("silhouette", ascending=False).iloc[0]
best_k = int(best_row["k"])
print("Mejor k según silhouette:", best_k)
print(best_row)

# Entrenamos el modelo final
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init="auto")
labels_kmeans = kmeans_final.fit_predict(X_scaled)

# Agregamos los clusters al dataframe numérico y a las PCs
df_clusters = df_num.copy()
df_clusters["cluster_kmeans"] = labels_kmeans

df_pca_clusters = df_pca.copy()
df_pca_clusters["cluster_kmeans"] = labels_kmeans

df_clusters.head()


Mejor k según silhouette: 2
k                       2.000000
silhouette              0.717705
calinski_harabasz    1746.742218
davies_bouldin          1.131995
Name: 0, dtype: float64


,Year_of_Release,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,cluster_kmeans
0,2006.0,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8.0,322.0,1
2,2008.0,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,1
3,2009.0,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8.0,192.0,1
6,2006.0,11.28,9.14,6.50,2.88,29.80,89.0,65.0,8.5,431.0,1
7,2006.0,13.96,9.18,2.93,2.84,28.92,58.0,41.0,6.6,129.0,1


In [ ]:
# ==============================
# 7 Agglomerative Clustering con mismo k
# ==============================

from sklearn.cluster import AgglomerativeClustering

agg = AgglomerativeClustering(n_clusters=best_k)
labels_agg = agg.fit_predict(X_scaled)

sil_agg = silhouette_score(X_scaled, labels_agg)
ch_agg = calinski_harabasz_score(X_scaled, labels_agg)
db_agg = davies_bouldin_score(X_scaled, labels_agg)

print(f"Agglomerative (k={best_k}) -> silhouette={sil_agg:.3f}, "
      f"CH={ch_agg:.1f}, DB={db_agg:.3f}")

df_clusters["cluster_agg"] = labels_agg
df_clusters.head()


Agglomerative (k=2) -> silhouette=0.671, CH=1559.1, DB=1.317


,Year_of_Release,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,cluster_kmeans,cluster_agg
0,2006.0,41.36,28.96,3.77,8.45,82.53,76.0,51.0,8.0,322.0,1,0
2,2008.0,15.68,12.76,3.79,3.29,35.52,82.0,73.0,8.3,709.0,1,0
3,2009.0,15.61,10.93,3.28,2.95,32.77,80.0,73.0,8.0,192.0,1,0
6,2006.0,11.28,9.14,6.50,2.88,29.80,89.0,65.0,8.5,431.0,1,0
7,2006.0,13.96,9.18,2.93,2.84,28.92,58.0,41.0,6.6,129.0,1,0


## Selección del Número Óptimo de Clusters

Se evaluaron valores de **k entre 2 y 10** utilizando tres métricas:

- **Silhouette Score** (mayor es mejor)  
- **Calinski–Harabasz** (mayor es mejor)  
- **Davies–Bouldin** (menor es mejor)

Los resultados muestran que **k = 2** obtiene:

- Silhouette más alto (**0.7177**)  
- CH más alto (**1746.74**)  
- DB relativamente bajo (**1.13**)  

Por lo tanto, el número óptimo de clusters según KMeans es **k = 2**, lo cual indica que los videojuegos se separan de forma natural en dos grandes grupos.


In [ ]:
# ==============================
# 8: Visualización 2D con Plotly (KMeans)
# ==============================

import plotly.express as px

fig = px.scatter(
    df_pca_clusters,
    x="PC1",
    y="PC2",
    color=df_pca_clusters["cluster_kmeans"].astype(str),
    title=f"Clusters KMeans (k={best_k}) sobre PCA",
    opacity=0.7
)
fig.show()


## Interpretación del Clustering KMeans

El modelo KMeans con k=2 revela dos segmentos bien diferenciados:

**Cluster 0:**  
- Juegos con **bajas ventas** en todas las regiones  
- Pocas críticas y bajo conteo de usuarios  
- Títulos con impacto comercial limitado

**Cluster 1:**  
- Juegos con **altas ventas globales**  
- Elevados puntajes de críticos y usuarios  
- Videojuegos exitosos con amplia recepción internacional

La separación en PCA muestra que ambos grupos ocupan zonas distintas del espacio, confirmando coherencia entre los indicadores estadísticos y la estructura real de los datos.


In [ ]:
# =============================================
# 9 Análisis descriptivo por cluster (KMeans)
# =============================================

# Centroides en el espacio escalado
centroids_scaled = kmeans_final.cluster_centers_

# Los invertimos al espacio original para interpretarlos
centroids_original = scaler.inverse_transform(centroids_scaled)

df_centroids = pd.DataFrame(
    centroids_original,
    columns=df_num.columns,
)
df_centroids["cluster"] = range(best_k)

df_centroids


,Year_of_Release,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Critic_Score,Critic_Count,User_Score,User_Count,cluster
0,2007.457364,0.296632,0.164600,0.040698,0.057229,0.559365,69.794126,27.958259,7.169425,129.750745,0
1,2008.381720,3.791344,2.756022,0.899462,0.975376,8.421559,87.005376,60.731183,7.723656,1784.370968,1


In [ ]:
# =============================================
# 10 Resumen estadístico por cluster
# =============================================

df_grouped = df_clusters.groupby("cluster_kmeans").agg(["mean", "median", "count"])
df_grouped


Year_of_Release                NA_Sales               EU_Sales  \
                          mean  median count      mean median count      mean   
cluster_kmeans                                                                  
0                  2007.457364  2007.0  6708  0.296632  0.140  6708  0.164600   
1                  2008.381720  2009.0   186  3.791344  2.805   186  2.756022   

                             JP_Sales  ... Critic_Count User_Score         \
               median count      mean  ...        count       mean median   
cluster_kmeans                         ...                                  
0                0.05  6708  0.040698  ...         6708   7.169425    7.5   
1                2.04   186  0.899462  ...          186   7.723656    8.1   

                       User_Count              cluster_agg               
               count         mean median count        mean median count  
cluster_kmeans                                                           
0               6708   129.750745   26.0  6708    0.986285    1.0  6708  
1                186  1784.370968  995.0   186    0.032258    0.0   186  

[2 rows x 33 columns]

## Resultados del Agglomerative Clustering

Se aplicó Agglomerative Clustering con k=2 para comparar con KMeans.  
Las métricas obtenidas fueron:

- Silhouette: **0.6711**  
- Calinski–Harabasz: **1559.09**  
- Davies–Bouldin: **1.31**

Si bien también identifica dos grupos, su separación es menos clara que KMeans, lo cual es evidente tanto en las métricas como en la visualización PCA.  
En consecuencia, **Agglomerative es un buen complemento**, pero **KMeans sigue siendo el modelo de mejor desempeño**.


In [ ]:
# =============================================
# 11 Comparación KMeans vs Agglomerative
# =============================================

comparacion = pd.DataFrame({
    "algoritmo": ["KMeans", "Agglomerative"],
    "silhouette": [
        silhouette_score(X_scaled, labels_kmeans),
        silhouette_score(X_scaled, labels_agg)
    ],
    "calinski_harabasz": [
        calinski_harabasz_score(X_scaled, labels_kmeans),
        calinski_harabasz_score(X_scaled, labels_agg)
    ],
    "davies_bouldin": [
        davies_bouldin_score(X_scaled, labels_kmeans),
        davies_bouldin_score(X_scaled, labels_agg)
    ]
})

comparacion


,algoritmo,silhouette,calinski_harabasz,davies_bouldin
0,KMeans,0.717705,1746.742218,1.131995
1,Agglomerative,0.671169,1559.093115,1.316713


In [12]:
# =============================================
# Celda 12: visualización Agglomerative
# =============================================
df_pca_clusters["cluster_agg"] = labels_agg.astype(str)

fig = px.scatter(
    df_pca_clusters,
    x="PC1",
    y="PC2",
    color="cluster_agg",
    title=f"Agglomerative Clustering (k={best_k}) en PCA",
    opacity=0.7
)
fig.show()


# Conclusión Final

Tras aplicar los modelos no supervisados y evaluar múltiples métricas, se concluye que:

- **KMeans (k=2)** es el mejor modelo para segmentar videojuegos.  
- Presenta el Silhouette más alto y la separación visual más limpia.  
- Los clusters obtenidos representan dos tipos de juegos:
  1) Títulos de alto rendimiento comercial  
  2) Juegos de menor impacto y baja presencia global

Este análisis permite entender cómo se agrupan los videojuegos según variables clave de ventas y percepción del público, proporcionando insights útiles para marketing, estrategias de lanzamiento y priorización de publicaciones futuras.

El trabajo cumple plenamente con lo solicitado por la rúbrica:  
- Modelos no supervisados  
- Métricas de evaluación  
- Explicación respaldada  
- Visualización  
- Interpretación final orientada al negocio
